# Create a Feature Encoding and Scaling Helper

Now that we have cleaned the dataset, the next step is preparing features for modeling.  
In this lesson we build a helper that identifies categorical columns, encodes them into numeric form, and scales numeric features when needed.

## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas scikit-learn python-dotenv

In [4]:
from cleaning_helper import cleaning_helper

import os
import re
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

df_raw = pd.read_csv("../../data/hr_analytics.csv")
df, diagnosis, code_clean = cleaning_helper(df_raw)


## 2 - Why Encoding Matters

Machine learning models work with numbers. Two problems need solving before training:

**Numeric scaling** — `MonthlyIncome` ranges from 1k to 20k, `age` from 18 to 65. Without scaling, high-magnitude features dominate linear models and distance-based models. Tree-based models don't need it, but it never hurts.

**Categorical encoding** — columns like `department` or `gender` contain strings. Models can't process them directly. The right strategy depends on the column:

| Strategy | When to use | Example |
|---|---|---|
| Binary map | Exactly 2 unique values | `OverTime`: Yes/No → 1/0 |
| Ordinal encoding | Categories have a meaningful order | `job_level`: 1,2,3,4,5 |
| One-hot encoding | Nominal, low cardinality (≤15 unique) | `department`: 6 values |
| Frequency encoding | Nominal, high cardinality (>15 unique) | `JobTitle`: 50+ values |
| Skip | Identifiers, targets, dates | `Employee ID`, `Attrition` |




## 3 - The Leakage Rule

> **Fit encoders and scalers on training data only. Transform test data using the fitted objects.**

Fitting on the full dataset leaks information about the test set. For example, `StandardScaler` computes mean and std across all rows and if test rows are included, the model has implicitly seen the test distribution before evaluation.

The correct pattern:
```python
encoder.fit(X_train)       # learn from training data only
X_train_enc = encoder.transform(X_train)
X_test_enc  = encoder.transform(X_test)   # apply same transform, no re-fitting
```

`build_encoder` returns a fitted `ColumnTransformer` so you can do exactly this.

## 4 - Build a Data Profile


Same profile pattern as Lesson 2.1 — one line per column with dtype, missing count, unique count, and sample values. The LLM uses this to decide encoding strategy per column.


In [5]:
def build_dataframe_profile(frame: pd.DataFrame) -> pd.DataFrame:
    """Build a compact profile of each column.

    Args:
        frame: Input DataFrame. Not modified in place.

    Returns:
        DataFrame with one row per column.
    """
    missing = frame.isna().sum()
    rows = []

    for col in frame.columns:
        series = frame[col]

        # top values with counts
        top_values = series.value_counts(dropna=True).head(5)
        top_values = [(str(v), int(c)) for v, c in top_values.items()]

        rows.append(
            {
                "column": col,
                "dtype": str(frame[col].dtype),
                "missing": int(missing[col]),
                "missing_pct": round(missing[col] / len(frame) * 100, 1),
                "unique": int(frame[col].nunique(dropna=True)),
                "samples": frame[col].dropna().head(10).tolist(),
            }
        )
    return pd.DataFrame(rows)


def profile_to_str(profile_df: pd.DataFrame) -> str:
    """Convert profile DataFrame to a string for the LLM."""
    lines = []
    for _, row in profile_df.iterrows():
        lines.append(
            f"{row.column} | dtype={row.dtype} | missing={row.missing} ({row.missing_pct}%) | unique={row.unique} | samples={row.samples}"
        )
    return "\n".join(lines)


In [6]:
# display profile — this is what the LLM will see
profile = build_dataframe_profile(df)
profile


,column,dtype,missing,missing_pct,unique,samples
0,Employee ID,int64,0,0.0,5000,"[1001, 1002, 1003, 1004, 1005, 1006, 1007, 100..."
1,age,int64,0,0.0,40,"[27, 34, 50, 31, 51, 39, 43, 44, 40, 40]"
2,gender,str,0,0.0,3,"[Female, Male, Female, Male, Male, Male, Femal..."
3,department,str,0,0.0,6,"[Engineering, Engineering, Finance, Marketing,..."
4,department_code,str,0,0.0,6,"[ENG-02, ENG-02, FIN-05, MKT-04, MKT-04, SAL-0..."
5,JobTitle,str,0,0.0,42,"[Backend Developer, Data Engineer, Controller,..."
6,job_level,int64,0,0.0,5,"[3, 2, 4, 3, 4, 3, 4, 4, 4, 1]"
7,Education,str,0,0.0,4,"[High School, Master's, Master's, Bachelor's, ..."
8,MonthlyIncome,int64,0,0.0,3680,"[8001, 8777, 14021, 6402, 9277, 8688, 8399, 93..."
9,monthly_rate,int64,0,0.0,3889,"[9162, 10301, 18470, 8805, 12125, 10249, 10360..."


## 5 - The Encoding Helper

Two steps, same philosophy as 2.1:

1. **Plan** — LLM reads the profile and returns a JSON encoding decision per column. No code, no exec. You inspect and override here.
2. **Implement** — LLM takes the approved plan and writes the `ColumnTransformer` code. You exec it.

**What the LLM does:** step 1 decides strategy, step 2 writes the boilerplate. You write neither. You inspect both.




In [7]:
PLAN_PROMPT = (
    "You are a feature engineering assistant. "
    "Given a dataset profile, return a JSON array of encoding decisions. "
    "Each item must have exactly three keys: column, strategy, reason. "
    "Valid strategies: skip, binary, ordinal, onehot, scale. "
    "Rules: "
    "identifier or target -> skip; "
    "2 unique values -> binary; "
    "ordered categories -> ordinal; "
    "nominal any cardinality -> onehot; "
    "continuous numeric -> scale; "
    "datetime -> skip. "
    "Return ONLY a valid JSON array. No explanation. No markdown fences."
)

IMPLEMENT_PROMPT = (
    "You are a senior machine learning engineer. "
    "Given a JSON encoding plan and a dataset profile, write executable Python code "
    "that builds and fits a ColumnTransformer on the existing DataFrame variable `X_train`. "
    "STRICT RULES: "
    "1. Use ONLY these objects: ColumnTransformer, OrdinalEncoder, OneHotEncoder, StandardScaler, pd, np. "
    "2. Do NOT use Pipeline, BaseEstimator, TransformerMixin, FunctionTransformer, or any custom classes. "
    "3. Never hardcode category lists. Use OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1). "
    "4. Use OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=15) for onehot. "
    "5. Do not define any functions or classes. "
    "6. Save the fitted ColumnTransformer to `encoder`. "
    "7. No imports. "
    "8. Always set remainder='passthrough' on the ColumnTransformer so that skipped columns are preserved as-is. "
    "Return only executable Python code."
)

In [8]:
def get_encoding_plan(
    frame: pd.DataFrame,
    target_col: str = None,
    column_types: dict = None,
    show_plan: bool = False,
):
    """Ask LLM for a JSON encoding plan. No code, no exec.

    Args:
        frame:        Input DataFrame.
        target_col:   Target column to skip.
        column_types: Manual overrides e.g. {"JobTitle": "frequency"}.
        show_plan:    If True, print the plan.

    Returns:
        list of dicts — [{column, strategy, reason}, ...]
    """
    work = frame.copy()
    if target_col and target_col in work.columns:
        work = work.drop(columns=[target_col])

    profile_str = profile_to_str(build_dataframe_profile(work))

    if column_types:
        overrides_str = "\n".join([f"  {k}: {v}" for k, v in column_types.items()])
        profile_str += f"\n\nColumn type overrides (use these strategies exactly):\n{overrides_str}"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=profile_str,
        config={"system_instruction": PLAN_PROMPT, "temperature": 0.0},
    )

    text = re.search(
        r"```(?:python|json)?\s*(.*?)```",
        response.text or "",
        flags=re.DOTALL | re.IGNORECASE,
    )
    text = text.group(1).strip() if text else (response.text or "").strip()
    plan = json.loads(text)

    if show_plan:
        print("--- Encoding Plan ---")
        for d in plan:
            print(
                f"  {d['column']:<30} strategy={d['strategy']:<12} reason={d['reason']}"
            )
        print()

    return plan

In [9]:
def build_encoder(plan: list, frame: pd.DataFrame, show_code: bool = False):
    """Take an approved encoding plan, ask LLM to write the ColumnTransformer code, exec it.

    Args:
        plan:      List of encoding decisions from get_encoding_plan.
        frame:     Training DataFrame to fit on.
        show_code: If True, print generated code before executing.

    Returns:
        (encoder, code) — fitted ColumnTransformer and generated code string.
    """
    profile_str = profile_to_str(build_dataframe_profile(frame))
    plan_str = json.dumps(plan, indent=2)

    prompt = (
        f"Dataset profile:\n{profile_str}\n\n"
        f"Encoding plan:\n{plan_str}\n\n"
        "DataFrame is available as `X_train`. Save fitted ColumnTransformer to `encoder`."
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={"system_instruction": IMPLEMENT_PROMPT, "temperature": 0.0},
    )

    text = response.text or ""
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text.strip()

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---\n")

    env = {
        "pd": pd,
        "np": np,
        "X_train": frame.copy(),
        "StandardScaler": StandardScaler,
        "OrdinalEncoder": OrdinalEncoder,
        "OneHotEncoder": OneHotEncoder,
        "ColumnTransformer": ColumnTransformer,
    }
    exec(code, env, env)
    encoder = env.get("encoder")
    if encoder is None:
        raise RuntimeError("Encoder code did not assign `encoder`.")
    return encoder, code

## 4 - Testing


In [10]:
X = df.drop(columns=["Attrition"])
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")


Train: (4000, 24)  Test: (1000, 24)


In [11]:
# step 1 : get encoding plan, inspect before any code runs
plan = get_encoding_plan(X_train, show_plan=True)

--- Encoding Plan ---
  Employee ID                    strategy=skip         reason=Identifier column with high cardinality.
  age                            strategy=scale        reason=Continuous numeric variable.
  gender                         strategy=onehot       reason=Nominal categorical variable with more than 2 unique values.
  department                     strategy=onehot       reason=Nominal categorical variable with more than 2 unique values.
  department_code                strategy=onehot       reason=Nominal categorical variable with more than 2 unique values.
  JobTitle                       strategy=onehot       reason=Nominal categorical variable with high cardinality.
  job_level                      strategy=ordinal      reason=Ordered categorical variable (e.g., 1 to 5).
  Education                      strategy=ordinal      reason=Ordered categorical variable (e.g., High School, Bachelor's, Master's).
  MonthlyIncome                  strategy=scale        reaso

In [12]:
# step 2 :build encoder from approved plan
encoder, code_enc = build_encoder(plan, X_train, show_code=True)

--- Generated Code ---
numerical_cols = [
    "age",
    "MonthlyIncome",
    "monthly_rate",
    "hourly_rate",
    "daily_rate",
    "years_at_company",
    "years_in_role",
    "years_since_promotion",
    "distance_from_home",
    "training_hours_last_year",
    "num_companies_worked"
]

onehot_cols = [
    "gender",
    "department",
    "department_code",
    "JobTitle"
]

ordinal_cols = [
    "job_level",
    "Education",
    "satisfaction_score",
    "environment_satisfaction",
    "manager_rating",
    "work_life_balance",
    "OverTime" # Binary variable handled by OrdinalEncoder as per rules
]

preprocessor_transformers = []

if numerical_cols:
    preprocessor_transformers.append(('num', StandardScaler(), numerical_cols))
if onehot_cols:
    preprocessor_transformers.append(('cat_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=15), onehot_cols))
if ordinal_cols:
    preprocessor_transformers.append(('cat_ordinal', OrdinalEncoder(handle_un

In [13]:
# transform train and test separately — no leakage
X_train_enc = encoder.transform(X_train)
X_test_enc = encoder.transform(X_test)

print(f"X_train encoded: {X_train_enc.shape}")
print(f"X_test  encoded: {X_test_enc.shape}")

X_train encoded: (4000, 50)
X_test  encoded: (1000, 50)


## 5 - Inspect and Override

Read the encoding plan. If any decisions look wrong, edit the plan yourself, then re-run Step 2 only to rebuild the encoder.

In [ ]:
# manual override example
# plan_corrected = [dict(item) for item in plan]
#
# for item in plan_corrected:
#     if item["column"] == "Employee ID":
#         item["strategy"] = "skip"
#         item["reason"] = "Identifier column"
#     elif item["column"] == "JobTitle":
#         item["strategy"] = "onehot"
#         item["reason"] = "Nominal categorical feature"
#
# encoder, code_enc = build_encoder(plan_corrected, X_train, show_code=True)
#
# X_train_enc = encoder.transform(X_train)
# X_test_enc  = encoder.transform(X_test)
#
# print(f"X_train encoded: {X_train_enc.shape}")
# print(f"X_test  encoded: {X_test_enc.shape}")


## 6 - Save for Reuse
Saved to `encoding_helper.py`. Module 3 imports `get_encoding_plan` and `build_encoder` directly.



In [14]:
import inspect

components = [
    "import os",
    "import re",
    "import json",
    "import numpy as np",
    "import pandas as pd",
    "from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder",
    "from sklearn.compose import ColumnTransformer",
    "from google import genai",
    "from dotenv import load_dotenv",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    inspect.getsource(build_dataframe_profile),
    "",
    inspect.getsource(profile_to_str),
    "",
    f"PLAN_PROMPT = {repr(PLAN_PROMPT)}",
    "",
    f"IMPLEMENT_PROMPT = {repr(IMPLEMENT_PROMPT)}",
    "",
    inspect.getsource(get_encoding_plan),
    "",
    inspect.getsource(build_encoder),
]

with open("encoding_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved encoding_helper.py")


Saved encoding_helper.py
